# AuthentiScan — A2: GPU smoke + per-architecture calibration

Group 148 | Semester 8 implementation track | plan: `sem8_major/implementation_plan.md` §7a A2

**Before Run All:** Settings → Accelerator = **GPU T4 x2** (or P100), Internet = **On**,
and Add-ons → Secrets must contain **`GITHUB_PAT`** (fine-grained token, read access to the
private `sm7` repo). CIFAKE must already be attached under Add Data.

This session does four things and nothing else:
1. record the exact environment (Paper 2 must report hardware and versions),
2. run the CPU-proven smoke config on GPU (proves the AMP path),
3. time all five backbones so A3 sessions can be sized against the session cap,
4. leave `results/` downloadable.

No matrix run happens here. Logic lives in `code/`, never in these cells.

In [ ]:
# 1. Fetch the code. Private repo via a Kaggle Secret; nothing sensitive is printed.
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT = UserSecretsClient().get_secret("GITHUB_PAT")
GITHUB_USER = "rohityaduvxnshi"   # <- change if the repo lives under a different account
REPO = "sm7"

if not os.path.exists("/kaggle/working/sm7"):
    subprocess.run(
        ["git", "clone", f"https://{PAT}@github.com/{GITHUB_USER}/{REPO}.git",
         "/kaggle/working/sm7"], check=True)
else:
    subprocess.run(["git", "-C", "/kaggle/working/sm7", "pull"], check=True)

os.chdir("/kaggle/working/sm7/sem8_major")
print("cwd:", os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# 2. Dependencies. Kaggle ships torch/torchvision with a matched CUDA build - never upgrade
# them here, or the session loses GPU support. Only the two extras are installed.
!pip install -q timm grad-cam
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Record the environment -> results/kaggle_env.md (commit this; Paper 2 quotes it)
!python code/record_env.py --out /kaggle/working/results/kaggle_env.md

In [ ]:
# 4. GPU smoke run: same config already proven on the laptop CPU, now with AMP on.
# Writes to smoke_runs.csv (train.py refuses to let a subset run touch runs.csv).
DATA = "/kaggle/input/cifake-real-and-ai-generated-synthetic-images"
RESULTS = "/kaggle/working/results"

!python code/train.py --config configs/resnet50_fe_smoke.yaml \
    --data-root {DATA} --results-dir {RESULTS} --amp true --num-workers 2

In [ ]:
# 5. Grad-CAM on the GPU checkpoint - proves the visualisation path end to end.
import csv, pathlib
rows = list(csv.DictReader(open(f"{RESULTS}/smoke_runs.csv", encoding="utf-8")))
SMOKE_RUN_ID = rows[-1]["run_id"]
print("smoke run:", SMOKE_RUN_ID, "| val_acc", rows[-1]["val_acc"], "| test_acc", rows[-1]["test_acc"])

!python code/gradcam.py --run-id {SMOKE_RUN_ID} --n 4 \
    --results-dir {RESULTS} --data-root {DATA} --out-root /kaggle/working/figures/gradcam

In [ ]:
# 6. Per-architecture calibration: times forward+backward for all five backbones in ft mode
# and projects full-size epochs. These numbers size every A3 session. ~10-15 min.
!python code/calibrate.py --data-root {DATA} --batches 30 \
    --out /kaggle/working/results/calibration.csv

In [ ]:
# 7. Session plan from the measured numbers. Cap assumption is conservative until the real
# single-session limit is confirmed (plan A2.4 [VERIFY]).
import pandas as pd

SESSION_CAP_H = 9.0      # [VERIFY] Kaggle free-tier single-session limit
MARGIN = 0.70            # plan rule: stay >=30% under the cap

cal = pd.read_csv("/kaggle/working/results/calibration.csv")
cal["fits_alone"] = cal["est_30_epoch_h"] < SESSION_CAP_H * MARGIN
display(cal[["model", "batch_size", "per_batch_s", "est_epoch_min", "est_30_epoch_h", "fits_alone"]])
print(f"usable session budget: {SESSION_CAP_H * MARGIN:.1f} h")
print("fe runs are cheaper than the ft numbers above (no backbone backward pass).")
print("VGG19-ft and ViT-ft run one per session by rule, regardless of these numbers.")

In [ ]:
# 8. Package the outputs for download (Output tab -> results_a2.zip).
!cd /kaggle/working && zip -qr results_a2.zip results figures && ls -la results_a2.zip
print("Download results_a2.zip, then locally:")
print("  python code/merge_runs.py <unzipped>/results/smoke_runs.csv --kind runs \\")
print("      --into results/smoke_runs.csv")
print("  (and copy kaggle_env.md + calibration.csv into sem8_major/results/, then commit)")